# Boltz2-Notebook: Diffusion-Based Protein–Ligand Structure Prediction & Affinity Analysis

**AI-powered structure prediction and binding affinity analysis** — Interactive Jupyter notebook for protein structure prediction, ligand binding, and multi-chain complex modeling using the Boltz2 diffusion model.

![Python](https://img.shields.io/badge/Python-3.10-blue?logo=python)
![CUDA](https://img.shields.io/badge/CUDA-Enabled-green?logo=nvidia)
![Boltz2](https://img.shields.io/badge/Model-Boltz2-purple)
![Platform](https://img.shields.io/badge/Platform-Colab%20-lightgrey?logo=googlecolab)


---

## Boltz2 version

**V1.0.0**  
Multi-chain proteins, protein-ligand binding, affinity analysis, 3D visualization  

---

## Overview

**Boltz2-Notebook** is an interactive Google Colab platform that integrates the Boltz2 deep learning model for diffusion-based structure prediction and binding affinity analysis. This notebook integrated the Boltz2 diffusion-based model to evaluate designed protein sequences in complex with small-molecule ligands. The notebook enables prediction of 3D protein–ligand structures, extraction of confidence metrics, and estimation of binding affinity for downstream comparison of protein variants.

---

## Workflow
1. **Setup** - Install dependencies and initialize Boltz2 environment with CUDA support for accelerated inference.
2. **Input Sequences** - Provide a set of protein sequences (generated_sequences) as input for evaluation.
3. **Ligand Definition** - Define ligand structures using SMILES representation (e.g. cAMP, cGMP).
4. **Configuration** - Generate YAML input files combining protein sequences and ligand definitions for Boltz2.
5. **Prediction** - Run Boltz2 diffusion-based inference to predict protein–ligand complex structures and binding properties.
6. **Analysis** - Extract confidence metrics and affinity predictions.
7. **Aggregation** - Compile all results into a structured CSV file for comparison across designs.
8. **Export** - DCollect predicted CIF structures and export them as ZIP files for visualization and external analysis.


In [ ]:
# @title Install Dependencies and Boltz2 with CUDA support
import sys
import subprocess
import threading
import time
import os
import shutil
import torch

os.chdir("/content/")

# ANSI color codes for colored output
class Color:
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    RESET = "\033[0m"

# ---------------- GPU CHECK ----------------
print(f"{Color.CYAN}[i] Checking GPU availability...{Color.RESET}")
if not torch.cuda.is_available():
    print(f"{Color.RED}[✘] No GPU detected!{Color.RESET}")
    print(f"{Color.YELLOW}Please change runtime to 'GPU' (T4 or higher).{Color.RESET}")
    print(f"{Color.CYAN}Runtime > Change Runtime Type > Select any available GPU from Hardware Accelerator.{Color.RESET}")
    sys.exit(1)
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"{Color.GREEN}[✔] GPU detected:{Color.RESET} {gpu_name}")

# ---------------- INSTALL STEPS ----------------
repo_dirs = ["boltz2-notebook"]

steps = [
    {
        "loader": f"{Color.CYAN}Cloning Notebook Modules...{Color.RESET}",
        "done":   f"[{Color.GREEN}✔{Color.RESET}] Notebook modules cloned successfully.",
        "fail":   f"[{Color.RED}✘{Color.RESET}] Boltz-Notebook clone failed.",
        "cmd": ["git", "clone", "https://github.com/AtharvaTilewale/boltz2-notebook.git"]
    },
]

def loader(msg, stop_event):
    symbols = ["-", "\\", "|", "/"]
    i = 0
    while not stop_event.is_set():
        sys.stdout.write(f"\r[{symbols[i % len(symbols)]}] {msg}   ")
        sys.stdout.flush()
        time.sleep(0.1)
        i += 1
    sys.stdout.write("\r" + " " * (len(msg) + 10) + "\r")

# Step 1: Remove repo if it exists
for repo in repo_dirs:
    if os.path.isdir(repo):
        print(f"{Color.YELLOW}[i] Repository already exists. Removing '{repo}'...{Color.RESET}")
        try:
            shutil.rmtree(repo)
            print(f"[{Color.GREEN}✔{Color.RESET}] Existing repository '{repo}' removed.")
        except Exception as e:
            print(f"[{Color.RED}✘{Color.RESET}] Failed to remove '{repo}': {e}")
            raise

all_success = True

# Main steps
for step in steps:
    stop_event = threading.Event()
    t = threading.Thread(target=loader, args=(step["loader"], stop_event))
    t.start()
    try:
        subprocess.run(step["cmd"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        stop_event.set()
        t.join()
        print(step["done"])
    except Exception as e:
        stop_event.set()
        t.join()
        print(f"{step['fail']} {e}")
        all_success = False
        break

# Run setup if clone worked
if all_success:
    %run /content/boltz2-notebook/scripts/v1/setup.py
    print(f"{Color.GREEN}All steps completed successfully.{Color.RESET}")

[i] Checking GPU availability...
[✔] GPU detected: Tesla T4
[✔] Notebook modules cloned successfully.
 ===Initialising Setup=== 


[✔] Boltz cloned successfully.
[✔] Dependencies installed successfully.
[✔] Validation complete.
All steps completed successfully.


# Input sequences
To predict designs with Boltz2, it is necessary to provide sequences on which the designs are based.

Generated sequences is a dictionary in which all sequences can be provided as input.

In [ ]:
generated_sequences = {'structure_1': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGELAIINSCPRSATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_2': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGELAVIMDRPRAATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_3': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGELACATSAARQATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_4': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIALLTGAPRAATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_5': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEFAALTGAPRSATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_6': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAVVTGAPRAATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_7': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEISLTTGAPRSATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_8': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAAATGKPRLATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_9': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIALETGKPRSATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_10': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAAAADQARAATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_11': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAALTDAPRSATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_12': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAALNGESAGATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_13': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAALLGAPRAADIILREDNCHFLRVDKQDFNRIIK',
                       'structure_14': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAALTGAARTATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_15': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAAASSSPRSATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_16': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAIATNAPRSATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_17': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIATLTDTPRSATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_18': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAIENDSPRTATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_19': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGQVAAITGAPRAATIILREDNCHFLRVDKQDFNRIIK',
                       'structure_20': 'EELAEAVALLSQRGPDALLTVALRKPPGQRTDEELDLIFEELLHIKAVAHLSNSVKRELAAVLLFEPHSKAGTVLFSQGDKGTSWYIIWKGSVNVVTHGKGLVTTLHEGDDFGEIAALTGLPRSATIILREDNCHFLRVDKQDFNRIIK'}

# Ligand SMILE code

The SMILE code can be used as input to add a ligand to the protein


In [ ]:
cAMP_smiles = "C1[C@@H]2[C@H]([C@H]([C@@H](O2)N3C=NC4=C(N=CN=C43)N)O)OP(=O)(O1)O"
cGMP_smiles = "C1[C@@H]2[C@H]([C@H]([C@@H](O2)N3C=NC4=C3N=C(NC4=O)N)O)OP(=O)(O1)O"

# Protein–Ligand Affinity Prediction Setup

This code sets up the working environment and input structure required to run Boltz2 protein–ligand affinity predictions. It defines a standardized YAML template that combines protein sequences with a ligand representation (SMILES format) and specifies an affinity prediction task. The script creates a data directory


In [ ]:
import os
import subprocess

class Color:
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    RESET = "\033[0m"

# create the data directory
os.makedirs("/content/boltz_data", exist_ok=True)
os.chdir("/content/boltz_data/")

# yaml template containing the protein sequence and ligand structure provided by the smiles code
# properties containing affinity is added to predict protein ligand affinity
yaml_template = """\
version: 1

sequences:
  - protein:
      id: [A]
      sequence: target_sequence
  - ligand:
      id: [L]
      smiles: cAMP_smiles

properties:
  - affinity:
      binder: L
"""

Starting run for structure_1 


FileNotFoundError: [Errno 2] No such file or directory: 'boltz'

# Running Boltz2

This code executes Boltz2 structure and binding affinity predictions for all generated protein sequences. For each sequence, a YAML input file and isolated output directory are created. Boltz2 is then run using MSA-based sequence alignment and potential-based scoring to estimate protein–ligand binding properties. The results are stored separately for each design to ensure traceable and non-overlapping outputs.


In [ ]:
# loop over all generated protein sequences
for name, sequence in generated_sequences.items():

    # insert current protein sequence into yaml template
    yaml_content = yaml_template.replace("target_sequence", sequence)

    # define ligand used for this prediction run
    # use a unique name to guarantee a a clean output map for each structure
    ligand_name = "cAMP_w_cAMP"

    # create unique yaml filename per design
    yaml_filename = f"predict_{name}_{ligand_name}.yaml"

    # create separate output directory for each design
    # ensures results are not overwritten between runs
    unique_output_dir = f"/content/boltz_data/clean_{name}_{ligand_name}"

    # define full path for yaml input file
    yaml_path = os.path.join("/content/boltz_data", yaml_filename)

    # write yaml input file for boltz
    with open(yaml_path, "w") as f:
        f.write(yaml_content)

    # print progress update for tracking runs
    print(f"{Color.GREEN}Starting run for {name} {Color.RESET}")

    # run boltz2 prediction pipeline
    # msa server provides evolutionary sequence context
    # potentials improve binding affinity estimation
    # output directory isolates each prediction result
    result = subprocess.run(
        [
            "boltz", "predict", yaml_filename,
            "--use_msa_server",
            "--use_potentials",
            "--out_dir", unique_output_dir
        ],
        cwd="/content/boltz_data",
        capture_output=True,
        text=True
    )

    # print completion status and return code for debugging
    # returncode = 0 indicates successful execution
    print(f"[{name}] prediction completed, Statuscode: {result.returncode}")


# Extracts and aggregating Boltz2 prediction outputs for all generated protein–ligand complexes.

Confidence metrics and affinity predictions are collected from the corresponding JSON output files for each design. The results are compiled into a structured CSV file for downstream analysis and comparison between protein variants.

In [ ]:
import csv
import json
import os
from google.colab import files

# define ligand name for this analysis run
# ensures to use the correct file paths and prevents mixing outputs between runs
# use the same name as the name provided in the Boltz2 run
ligand_name = "cAMP_w_cAMP"

# define CSV column headers for extracted Boltz2 metrics
headers = [
    "Structure", "Overall Confidence Score", "Complex pLDDT", "Complex ipLDDT",
    "Ligand iPTM", "Protein iPTM", "Complex pDE", "Complex ipDE",
    "Protein (Chain A) pTM", "Ligand (Chain L) pTM",
    "Protein-Protein iPTM", "Protein-Ligand iPTM", "Ligand-Protein iPTM", "Ligand-Ligand iPTM",
    "Affinity (log10 IC50 µM)", "Binder Probability", "Binder Probability (%)"
]

rows = []

# loop over all protein sequences
for name in generated_sequences.keys():

    # construct base path to Boltz2 output files
    base_path = f"/content/boltz_data/clean_{name}_{ligand_name}/boltz_results_predict_{name}_{ligand_name}/predictions/predict_{name}_{ligand_name}"

    # path to confidence output file
    conf_path = f"{base_path}/confidence_predict_{name}_{ligand_name}_model_0.json"

    # path to affinity prediction file
    aff_path = f"{base_path}/affinity_predict_{name}_{ligand_name}.json"

    # initialize empty row for current structure
    row_data = {}

    # fill all headers with empty values
    for h in headers:
        row_data[h] = ""

    # store structure name
    row_data["Structure"] = name

    # confidence data extraction
    if os.path.exists(conf_path):
        with open(conf_path) as f:
            conf = json.load(f)
        row_data["Overall Confidence Score"] = conf.get("confidence_score", "")
        row_data["Complex pLDDT"] = conf.get("complex_plddt", "")
        row_data["Complex ipLDDT"] = conf.get("complex_iplddt", "")
        row_data["Ligand iPTM"] = conf.get("ligand_iptm", "")
        row_data["Protein iPTM"] = conf.get("protein_iptm", "")
        row_data["Complex pDE"] = conf.get("complex_pde", "")
        row_data["Complex ipDE"] = conf.get("complex_ipde", "")

        # chain-specific pTM scores
        chains = conf.get("chains_ptm", {})
        row_data["Protein (Chain A) pTM"] = chains.get("0", "")
        row_data["Ligand (Chain L) pTM"] = chains.get("1", "")

        # pairwise interaction iPTM scores
        pair = conf.get("pair_chains_iptm", {})
        row_data["Protein-Protein iPTM"] = pair.get("0", {}).get("0", "")
        row_data["Protein-Ligand iPTM"] = pair.get("0", {}).get("1", "")
        row_data["Ligand-Protein iPTM"] = pair.get("1", {}).get("0", "")
        row_data["Ligand-Ligand iPTM"] = pair.get("1", {}).get("1", "")

    # affinity data extraction
    if os.path.exists(aff_path):
        with open(aff_path) as f:
            aff = json.load(f)
        row_data["Affinity (log10 IC50 µM)"] = aff.get("affinity_pred_value", "")
        prob = aff.get("affinity_probability_binary")
        if prob is not None:
            row_data["Binder Probability"] = prob
            row_data["Binder Probability (%)"] = prob * 100

    # add row to final dataset
    rows.append(row_data)

# save results as CSV file for downstream analysis
output_file = f"/content/boltz_data/boltz_loop_results_{ligand_name}.csv"
with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(rows)

# download csv file
files.download(output_file)

Gelukt! Alle unieke data voor cAMP_w_cGMP is verzameld. De download start...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Collecting predicted Boltz2 protein–ligand structures in CIF format for downstream visualization and analysis.

For each design, the corresponding structure file is located in the Boltz2 output directory and copied into a dedicated export folder. The collected structures are then compressed into a ZIP archive for easy download.


In [ ]:
import os
import shutil
from google.colab import files

# define ligand name for this export run
# ensures correct file paths and prevents mixing outputs between runs
# use the same name as the name provided in the Boltz2 run
ligand_name = "cAMP_w_cAMP"

# create temporary directory to store exported CIF files
export_dir = f"/content/boltz_cif_export_{ligand_name}"
os.makedirs(export_dir, exist_ok=True)

# counter to track number of successfully copied structures
count = 0

# loop over all protein designs
for name in generated_sequences.keys():

    # path to predicted CIF file from Boltz2 output
    source_cif = f"/content/boltz_data/clean_{name}_{ligand_name}/boltz_results_predict_{name}_{ligand_name}/predictions/predict_{name}_{ligand_name}/predict_{name}_{ligand_name}_model_0.cif"

    # target path in export directory
    target_cif = os.path.join(export_dir, f"{name}_model_0.cif")

    # check if CIF file exists before copying
    if os.path.exists(source_cif):

        # copy structure file to export folder
        shutil.copy(source_cif, target_cif)

        # increase counter for successful copies
        count += 1

    else:
        # warning if file is missing for a specific design
        print(f"File {name} ({ligand_name}) not found on:\n{source_cif}\n")

# create zip archive for easy download and external visualization
zip_file_path = f"/content/boltz_models_{ligand_name}_for_pymol"
shutil.make_archive(zip_file_path, 'zip', export_dir)

# download zip file
files.download(f"{zip_file_path}.zip")

Er zijn 20 .cif-bestanden succesvol verzameld voor cAMP_w_cGMP!
Het ZIP-bestand voor cAMP_w_cGMP is gemaakt. De download start nu...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Credits

**Developed by:** Atharva Tilewale & Dr. Dhaval Patel  
**Affiliation:** Gujarat Biotechnology University | Bioinformatics & Computational Biology  
**Repository:** [github.com/AtharvaTilewale/boltz2-notebook](https://github.com/AtharvaTilewale/boltz2-notebook)

---

## Citation


**Boltz-2:**

[![bioRxiv Boltz2](https://img.shields.io/badge/bioRxiv-Boltz2-red)](https://doi.org/10.1101/2025.06.14.659707)

```
Passaro, S., Corso, G., Wohlwend, J., et al. (2025).
Boltz-2: Towards Accurate and Efficient Binding Affinity Prediction.
bioRxiv. https://doi.org/10.1101/2025.06.14.659707
```

**Boltz-1:**

[![bioRxiv Boltz1](https://img.shields.io/badge/bioRxiv-Boltz1-orange)](https://doi.org/10.1101/2024.11.19.624167)

```
Wohlwend, J., Corso, G., Passaro, S., et al. (2024).
Boltz-1: Democratizing Biomolecular Interaction Modeling.
bioRxiv. https://doi.org/10.1101/2024.11.19.624167
```

---

## License

MIT License — See [LICENSE](https://github.com/AtharvaTilewale/boltz2-notebook/blob/main/LICENSE) for details.

---